# PCINet Improved Training (fixed train/val/test split)

**Changes from the previous run:**
- Early stopping (patience 15)
- Dropout increased to 0.4 (stronger regularisation in the classifier head)
- Weight decay increased to 1e-4 (penalises large weights harder)
- ReduceLROnPlateau replaces CosineAnnealingLR
- MAE and soft accuracy (±1) reported and saved in test_metrics.csv 
- test_predictions.csv includes AE per image for GradCAM grouping
- Class weights computed via sklearn compute_class_weight
- WeightedRandomSampler uses sklearn weights directly
- confusion_matrices.txt saved with per-epoch and test confusion matrices

In [1]:
import os, sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms
import torchvision.transforms.functional as TF
from sklearn.metrics import (
    confusion_matrix, balanced_accuracy_score, classification_report,
    accuracy_score, cohen_kappa_score, f1_score, top_k_accuracy_score
)
from sklearn.utils.class_weight import compute_class_weight
from PIL import Image
import matplotlib.pyplot as plt
np.set_printoptions(threshold=sys.maxsize)

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')

Using device: cuda:0
GPU: Tesla T4


In [ ]:
# Path to dataset
DATASET_DIR = '/kaggle/input/datasets/.../.../.../'
IMAGES_DIR  = f'{DATASET_DIR}/images'
TRAIN_CSV   = f'{DATASET_DIR}/train.csv'
VAL_CSV     = f'{DATASET_DIR}/val.csv'
TEST_CSV    = f'{DATASET_DIR}/test.csv'

# Path to the output directory
OUTPUT_DIR  = '/kaggle/working'

# Path to the best saved model in case retraining is not needed
SAVE_PATH = '/kaggle/input/datasets/.../.../.../best_model.pt'

# Path to image annotations
ANNOT_PATH  = '/kaggle/input/datasets/.../.../annotations.csv'

# Checking if the files needed from the dataset exist
for path, name in [(IMAGES_DIR,'IMAGES_DIR'),(TRAIN_CSV,'TRAIN_CSV'),(VAL_CSV,'VAL_CSV'),(TEST_CSV,'TEST_CSV')]:
    print(f'{"checkmark" if os.path.exists(path) else "NOT FOUND"}  {name}')

# Hyperparameter settings
EPOCHS       = 75
PATIENCE     = 15
BATCH_SIZE   = 16
LR           = 1e-5
WEIGHT_DECAY = 1e-4
FOCAL_GAMMA  = 2
DROPOUT      = 0.4

# Set to False for retraining
SKIP_TRAINING = True

print('SKIP_TRAINING =', SKIP_TRAINING)
print('SAVE_PATH exists:', os.path.exists(SAVE_PATH))


checkmark  IMAGES_DIR
checkmark  TRAIN_CSV
checkmark  VAL_CSV
checkmark  TEST_CSV
SKIP_TRAINING = True
SAVE_PATH exists: True


## Dataset

In [ ]:
class MyRotationTransform:
    def __init__(self, angles):
        self.angles = angles
    def __call__(self, x):
        angle = np.random.choice(self.angles, p=[0.2, 0.6, 0.2])
        return TF.rotate(x, int(angle))


class CSVDataset(Dataset):
    def __init__(self, csv_path, images_dir, transform=None):
        self.df = pd.read_csv(csv_path)
        self.images_dir = images_dir
        self.transform = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(os.path.join(self.images_dir, row['name'])).convert('RGB')
        label = int(row['label'])
        if self.transform: image = self.transform(image)
        return image, label

    def class_counts(self):
        return self.df['label'].value_counts().sort_index().tolist()


mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize((384, 384)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomVerticalFlip(0.1),
    MyRotationTransform(angles=[-90, 0, 90]),
    transforms.GaussianBlur(kernel_size=(5, 9), sigma=(0.1, 3.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

eval_transforms = transforms.Compose([
    transforms.Resize((384, 384)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

train_dataset = CSVDataset(TRAIN_CSV, IMAGES_DIR, transform=train_transforms)
val_dataset   = CSVDataset(VAL_CSV,   IMAGES_DIR, transform=eval_transforms)
test_dataset  = CSVDataset(TEST_CSV,  IMAGES_DIR, transform=eval_transforms)

#  Class weights via sklearn compute_class_weight to handle imbalanced dataset
train_labels_array = train_dataset.df['label'].values
CLASS_WEIGHTS = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1, 2, 3, 4]),
    y=train_labels_array
).tolist()

#  WeightedRandomSampler using sklearn weights
sample_weights = np.array([CLASS_WEIGHTS[int(l)] for l in train_labels_array])
train_sampler = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(sample_weights), replacement=True
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=train_sampler,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

counts = np.array(train_dataset.class_counts(), dtype=np.float32)
print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')
print('Train class counts (0-4):', counts.astype(int).tolist())
print('Class weights (sklearn): ', [f'{w:.3f}' for w in CLASS_WEIGHTS])

Train: 3755 | Val: 804 | Test: 806
Train class counts (0-4): [113, 750, 1222, 1170, 500]
Class weights (sklearn):  ['6.646', '1.001', '0.615', '0.642', '1.502']


## Metrics

In [4]:
# Defining the metrics
def calculate_metrics(preds, labels, scores, split='Val'):
    preds, labels = np.asarray(preds), np.asarray(labels)
    scores = np.asarray(scores)

    cm       = confusion_matrix(labels, preds, normalize='true')
    b_acc    = balanced_accuracy_score(labels, preds)
    acc      = accuracy_score(labels, preds)
    kappa    = cohen_kappa_score(labels, preds)
    f1       = f1_score(labels, preds, average='weighted')
    top2     = top_k_accuracy_score(labels, scores, k=2)
    top3     = top_k_accuracy_score(labels, scores, k=3)
    mae      = np.abs(preds - labels).mean()
    soft_acc = (np.abs(preds - labels) <= 1).mean()

    print(f'\n[{split}] Acc: {acc:.4f} | Bal Acc: {b_acc:.4f} | Kappa: {kappa:.4f} | '
          f'F1: {f1:.4f} | Top-2: {top2:.4f} | Top-3: {top3:.4f}')
    print(f'         MAE: {mae:.4f} | Soft Acc (±1): {soft_acc:.4f}  (paper: MAE=0.66, soft=0.8973)')
    print(cm)
    print(classification_report(labels, preds, zero_division=0))

    return {
        'accuracy': acc, 'balanced_accuracy': b_acc, 'kappa': kappa, 'f1': f1,
        'top_2_accuracy': top2, 'top_3_accuracy': top3,
        'mae': mae, 'soft_accuracy': soft_acc, 'confusion_matrix': cm
    }

## Focal Loss

In [5]:
# Defining the loss function
class FocalLoss(nn.CrossEntropyLoss):
    def __init__(self, gamma, alpha=None, ignore_index=-100, reduction='none'):
        super().__init__(weight=alpha, ignore_index=ignore_index, reduction='none')
        self.reduction = reduction
        self.gamma = gamma

    def forward(self, input_, target):
        cross_entropy = super().forward(input_, target)
        target = target * (target != self.ignore_index).long()
        input_prob = torch.gather(F.softmax(input_, 1), 1, target.unsqueeze(1))
        loss = torch.pow(1 - input_prob, self.gamma) * cross_entropy
        if self.reduction == 'mean': return torch.mean(loss)
        elif self.reduction == 'sum': return torch.sum(loss)
        return loss

## Model setup

In [6]:
# Defining the model(Inherited from the original PCINet paper)
model = models.efficientnet_v2_s(weights='IMAGENET1K_V1')
num_ftrs = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=DROPOUT, inplace=True),
    nn.Linear(in_features=num_ftrs, out_features=5, bias=True),
)
model.to(device)

class_weights_tensor = torch.FloatTensor(CLASS_WEIGHTS).to(device)
criterion = FocalLoss(gamma=FOCAL_GAMMA, alpha=class_weights_tensor, reduction='mean').to(device)
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=7, min_lr=1e-8
)

print(model.classifier)
print(f'Dropout: {DROPOUT} | Weight decay: {WEIGHT_DECAY} | Early stopping patience: {PATIENCE}')

Downloading: "https://download.pytorch.org/models/efficientnet_v2_s-dd5fe13b.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_s-dd5fe13b.pth


100%|██████████| 82.7M/82.7M [00:00<00:00, 195MB/s]


Sequential(
  (0): Dropout(p=0.4, inplace=True)
  (1): Linear(in_features=1280, out_features=5, bias=True)
)
Dropout: 0.4 | Weight decay: 0.0001 | Early stopping patience: 15


## Training loop with early stopping

## Test evaluation

In [ ]:
model.load_state_dict(torch.load(SAVE_PATH, map_location=device, weights_only=True))
model.eval()

test_pred, test_score, test_labels = [], [], []
with torch.no_grad():
    for inps, labs in test_loader:
        inps, labs = inps.to(device), labs.to(device)
        out = model(inps)
        test_pred   += out.max(1)[1].cpu().numpy().tolist()
        test_score  += out.detach().cpu().numpy().tolist()
        test_labels += labs.cpu().numpy().tolist()

test_metrics = calculate_metrics(test_pred, test_labels, test_score, split='Test')

pd.DataFrame([{k: v for k, v in test_metrics.items() if k != 'confusion_matrix'}]).to_csv(
    f'{OUTPUT_DIR}/test_metrics.csv', index=False)
print('\n Test metrics saved')


[Test] Acc: 0.2320 | Bal Acc: 0.3367 | Kappa: 0.0659 | F1: 0.2259 | Top-2: 0.4429 | Top-3: 0.6514
         MAE: 1.3213 | Soft Acc (±1): 0.6303  (paper: MAE=0.66, soft=0.8973)
[[0.76       0.         0.04       0.16       0.04      ]
 [0.42236025 0.0931677  0.13664596 0.17391304 0.17391304]
 [0.2519084  0.0648855  0.09541985 0.36641221 0.22137405]
 [0.2310757  0.05976096 0.06374502 0.34262948 0.30278884]
 [0.22429907 0.07476636 0.06542056 0.24299065 0.39252336]]
              precision    recall  f1-score   support

           0       0.08      0.76      0.15        25
           1       0.27      0.09      0.14       161
           2       0.35      0.10      0.15       262
           3       0.36      0.34      0.35       251
           4       0.20      0.39      0.27       107

    accuracy                           0.23       806
   macro avg       0.25      0.34      0.21       806
weighted avg       0.31      0.23      0.23       806


✓ Test metrics saved


## Save test predictions

In [ ]:
test_df = pd.read_csv(TEST_CSV)
test_df['pred'] = test_pred
test_df['ae']   = np.abs(np.array(test_pred) - np.array(test_labels))
for c in range(5):
    test_df[f'score_{c}'] = [s[c] for s in test_score]

test_df.to_csv(f'{OUTPUT_DIR}/test_predictions.csv', index=False)
print(' test_predictions.csv saved')

print('\nAE group breakdown:')
for ae_val, label in [(0,'AE=0 (correct)'),(1,'AE=1 (near-miss)')]:
    n = (test_df['ae'] == ae_val).sum()
    print(f'  {label}: {n} ({100*n/len(test_df):.1f}%)')
n = (test_df['ae'] > 1).sum()
print(f'  AE>1 (harsh error): {n} ({100*n/len(test_df):.1f}%)')

✓ test_predictions.csv saved

AE group breakdown:
  AE=0 (correct): 187 (23.2%)
  AE=1 (near-miss): 321 (39.8%)
  AE>1 (harsh error): 298 (37.0%)


## Save confusion matrices (training + test)

In [ ]:
if not SKIP_TRAINING:
    # Creating a readable confusion matrix
    def format_cm(norm_cm, support, title):
        lines = []
        lines.append(title)
        lines.append('-' * len(title))
    
        # Header
        header = f'  {"True\\Pred":<12}' + ''.join(f'  {"PCI " + str(j):<10}' for j in range(5))
        lines.append(header)
        lines.append('  ' + '-' * (len(header) - 2))
    
        for i in range(5):
            row_support = support[i] if support is not None else None
            row_str = f'  PCI {i}'
            if row_support is not None:
                row_str += f' (n={row_support:<4})'
            else:
                row_str = f'  PCI {i:<10}'
            for j in range(5):
                pct   = norm_cm[i][j] * 100
                count = round(norm_cm[i][j] * support[i]) if support is not None else ''
                cell  = f'{pct:.0f}% ({count})' if support is not None else f'{pct:.0f}%'
                row_str += f'  {cell:<10}'
            lines.append(row_str)
    
        return '\n'.join(lines)
    
    
    test_support = np.array([
        (np.array(test_labels) == c).sum() for c in range(5)
    ], dtype=int)
    
    out_path = f'{OUTPUT_DIR}/confusion_matrices.txt'
    with open(out_path, 'w') as f:
    
        f.write('=' * 70 + '\n')
        f.write('CONFUSION MATRICES PCINet Training Run\n')
        f.write('Each cell: % of true class predicted as that column class (count)\n')
        f.write('Diagonal cells = correct predictions\n')
        f.write('=' * 70 + '\n\n')
    
        #  Per epoch training and validation matrix 
        for rec in epoch_cms:
            ep = rec['epoch']
            marker = ' *** BEST CHECKPOINT ***' if rec['val_bacc'] == best_val_bacc else ''
            f.write(f'{"=" * 70}\n')
            f.write(f'EPOCH {ep}  |  Val Balanced Acc: {rec["val_bacc"]:.4f}  |  Val MAE: {rec["val_mae"]:.4f}{marker}\n')
            f.write(f'{"=" * 70}\n\n')
    
            f.write(format_cm(rec['train_cm'], train_support, 'Training Set CM'))
            f.write('\n\n')
            f.write(format_cm(rec['val_cm'], val_support, 'Validation Set CM'))
            f.write('\n\n')
    
        #  Test Confusion Matrix 
        f.write('=' * 70 + '\n')
        f.write('FINAL TEST SET CONFUSION MATRIX\n')
        f.write(f'Acc: {test_metrics["accuracy"]:.4f}  |  Bal Acc: {test_metrics["balanced_accuracy"]:.4f}  |  '
                f'MAE: {test_metrics["mae"]:.4f}  |  Soft Acc: {test_metrics["soft_accuracy"]:.4f}\n')
        f.write('=' * 70 + '\n\n')
        f.write(format_cm(test_metrics['confusion_matrix'], test_support, 'Test Set CM'))
        f.write('\n')
    
    print(f' confusion_matrices.txt saved to {out_path}')
    print(f'  {len(epoch_cms)} training epochs + 1 test run')
else:
    print("Confusion matrices skipped (SKIP_TRAINING=True). Already saved from previous run.")


Confusion matrices skipped (SKIP_TRAINING=True). Already saved from previous run.


## Training curves

In [ ]:
# Creating training curves to make monitoring the training easier
if not SKIP_TRAINING:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    eps = range(1, len(history['train_loss']) + 1)
    
    axes[0].plot(eps, history['train_loss'], label='train', color='#378ADD')
    axes[0].plot(eps, history['val_loss'],   label='val',   color='#1D9E75')
    axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch')
    axes[0].legend(); axes[0].grid(alpha=0.3)
    
    axes[1].plot(eps, history['train_bacc'], label='train', color='#378ADD')
    axes[1].plot(eps, history['val_bacc'],   label='val',   color='#1D9E75')
    axes[1].axhline(best_val_bacc, color='red', linestyle='--', alpha=0.5,
                    label=f'best={best_val_bacc:.3f}')
    axes[1].set_title('Balanced Accuracy'); axes[1].set_xlabel('Epoch')
    axes[1].legend(); axes[1].grid(alpha=0.3)
    
    axes[2].plot(eps, history['val_mae'],      label='MAE',      color='#D85A30')
    axes[2].plot(eps, history['val_soft_acc'], label='Soft acc', color='#534AB7')
    axes[2].axhline(0.66,   color='#D85A30', linestyle='--', alpha=0.4, label='paper MAE 0.66')
    axes[2].axhline(0.8973, color='#534AB7', linestyle='--', alpha=0.4, label='paper soft 0.897')
    axes[2].set_title('MAE & Soft Accuracy vs paper'); axes[2].set_xlabel('Epoch')
    axes[2].legend(fontsize=8); axes[2].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/training_curves.pdf', bbox_inches='tight')
    plt.show()
    print(' Training curves saved')
else:
    print("Training curves skipped (SKIP_TRAINING=True). Already saved from previous run.")


Training curves skipped (SKIP_TRAINING=True). Already saved from previous run.


## GradCAM Analysis

Generates GradCAM heatmaps for all test images with prediction errors (AE ≠ 0), using the [`pytorch-grad-cam`] library.

- Target layer: `model.features[-1]` (last convolutional block of EfficientNetV2-S)
- Heatmap target: predicted class
- AE=1 heatmaps saved to `gradcam_ae1/`
- AE>1 heatmaps saved to `gradcam_ae_gt1/`
- Filename: `<original_name>_gradcam.<ext>`

In [ ]:
!pip install grad-cam -q

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

In [ ]:
import cv2

# Load annotations
annot_df = pd.read_csv(ANNOT_PATH)
annot_lookup = {}
for _, arow in annot_df.iterrows():
    nm = arow['image_name']
    if nm not in annot_lookup:
        annot_lookup[nm] = []
    annot_lookup[nm].append((arow['box_type'], int(arow['xmin']), int(arow['ymin']),
                              int(arow['xmax']), int(arow['ymax'])))
print('Annotations loaded:', len(annot_df), 'boxes across', len(annot_lookup), 'images')

# Reload best checkpoint
model.load_state_dict(torch.load(SAVE_PATH, map_location=device, weights_only=True))
model.eval()

# Output directories
ae1_dir    = OUTPUT_DIR + '/gradcam_ae1'
ae_gt1_dir = OUTPUT_DIR + '/gradcam_ae_gt1'
os.makedirs(ae1_dir,    exist_ok=True)
os.makedirs(ae_gt1_dir, exist_ok=True)

# Load predictions, keep only errors
pred_df   = pd.read_csv(OUTPUT_DIR + '/test_predictions.csv')
errors_df = pred_df[pred_df['ae'] != 0].reset_index(drop=True)
print('Images to process:', len(errors_df))
print('  AE=1 :', (errors_df['ae'] == 1).sum())
print('  AE>1 :', (errors_df['ae'] > 1).sum())

# Targetting the last convolutional layer block of the model
cam = GradCAM(model=model, target_layers=[model.features[-1]])
n   = len(errors_df)
pg_results = []

for i, row in errors_df.iterrows():
    pil_img        = Image.open(os.path.join(IMAGES_DIR, row['name'])).convert('RGB')
    orig_w, orig_h = pil_img.size
    input_tensor   = eval_transforms(pil_img).unsqueeze(0).to(device)

    grayscale_cam = cam(input_tensor=input_tensor,
                        targets=[ClassifierOutputTarget(int(row['pred']))])

    # Upsample heatmap to original resolution
    heatmap_full = cv2.resize(grayscale_cam[0], (orig_w, orig_h))

    # Save heatmap visualization
    rgb_orig      = np.array(pil_img, dtype=np.float32) / 255.0
    visualization = show_cam_on_image(rgb_orig, heatmap_full, use_rgb=True)
    stem, ext = os.path.splitext(row['name'])
    out_name  = stem + '_gradcam' + (ext if ext else '.jpg')
    out_dir   = ae1_dir if row['ae'] == 1 else ae_gt1_dir
    Image.fromarray(visualization).save(os.path.join(out_dir, out_name))

    # Pointing Game
    peak_y, peak_x = np.unravel_index(np.argmax(heatmap_full), heatmap_full.shape)
    boxes         = annot_lookup.get(row['name'], [])
    in_relevant   = any(bt == 'relevant'   and x1 <= peak_x <= x2 and y1 <= peak_y <= y2
                        for bt, x1, y1, x2, y2 in boxes)
    in_irrelevant = any(bt == 'irrelevant' and x1 <= peak_x <= x2 and y1 <= peak_y <= y2
                        for bt, x1, y1, x2, y2 in boxes)
    pg_results.append({
        'name':          row['name'],
        'label':         int(row['label']),
        'pred':          int(row['pred']),
        'ae':            int(row['ae']),
        'peak_x':        int(peak_x),
        'peak_y':        int(peak_y),
        'in_relevant':   in_relevant,
        'in_irrelevant': in_irrelevant,
        'hit':           in_relevant and not in_irrelevant,
    })

    if (i + 1) % 50 == 0 or (i + 1) == n:
        print(str(i + 1) + '/' + str(n), end='  ', flush=True)

print('')
print('Done.')
results_df = pd.DataFrame(pg_results)
results_df.to_csv(OUTPUT_DIR + '/pointing_game_results.csv', index=False)
print('Saved: pointing_game_results.csv')
print('  AE=1  heatmaps ->', ae1_dir)
print('  AE>1  heatmaps ->', ae_gt1_dir)


In [ ]:
import shutil
shutil.make_archive(OUTPUT_DIR + '/gradcam_ae1',    'zip', ae1_dir)
shutil.make_archive(OUTPUT_DIR + '/gradcam_ae_gt1', 'zip', ae_gt1_dir)
print('Done')

## Pointing Game Analysis

Evaluates whether the GradCAM peak activation falls on a relevant region, on an irrelevant object or on the background.

- Hit: peak is inside a relevant box AND NOT inside any irrelevant box
- Miss: peak is on an irrelevant object
- Background: on background


In [ ]:
print('POINTING GAME RESULTS')

groups = [
    ('AE=1  (near-miss)',   results_df['ae'] == 1),
    ('AE>1  (harsh error)', results_df['ae'] >  1),
    ('All errors (AE!=0)',  results_df['ae'] >  0),
]

for label, mask in groups:
    sub  = results_df[mask]
    n    = len(sub)
    hits = sub['hit'].sum()
    irr  = sub['in_irrelevant'].sum()
    bg   = ((~sub['in_relevant']) & (~sub['in_irrelevant'])).sum()
    print(f'\n{label}  (n={n})')
    print(f'  Hit rate (relevant, not distracted) : {100*hits/n:.1f}%  ({hits}/{n})')
    print(f'  Peak on irrelevant object (car/person): {100*irr/n:.1f}%  ({irr}/{n})')
    print(f'  Peak on background (sky/trees/other)  : {100*bg/n:.1f}%  ({bg}/{n})')


POINTING GAME RESULTS

AE=1  (near-miss)  (n=321)
  Hit rate (relevant, not distracted) : 71.3%  (229/321)
  Peak on irrelevant object (car/person): 2.5%  (8/321)
  Peak on background (sky/trees/other)  : 26.2%  (84/321)

AE>1  (harsh error)  (n=298)
  Hit rate (relevant, not distracted) : 65.8%  (196/298)
  Peak on irrelevant object (car/person): 3.0%  (9/298)
  Peak on background (sky/trees/other)  : 31.2%  (93/298)

All errors (AE!=0)  (n=619)
  Hit rate (relevant, not distracted) : 68.7%  (425/619)
  Peak on irrelevant object (car/person): 2.7%  (17/619)
  Peak on background (sky/trees/other)  : 28.6%  (177/619)


In [ ]:
# Saving the images
from PIL import ImageDraw
import zipfile

debug_dir = OUTPUT_DIR + '/debug_overlays'
os.makedirs(debug_dir, exist_ok=True)

for _, row in results_df.iterrows():
    stem, ext = os.path.splitext(row['name'])
    gradcam_name = stem + '_gradcam' + (ext if ext else '.jpg')
    src_dir = ae1_dir if row['ae'] == 1 else ae_gt1_dir
    gradcam_path = os.path.join(src_dir, gradcam_name)
    if not os.path.exists(gradcam_path):
        continue

    img  = Image.open(gradcam_path).convert('RGB')
    draw = ImageDraw.Draw(img)

    for bt, x1, y1, x2, y2 in annot_lookup.get(row['name'], []):
        draw.rectangle([x1, y1, x2, y2],
                       outline='green' if bt == 'relevant' else 'red', width=3)

    px, py, r = int(row['peak_x']), int(row['peak_y']), 15
    draw.ellipse([px-r, py-r, px+r, py+r], outline='yellow', width=3)
    draw.line([px-r*2, py, px+r*2, py], fill='yellow', width=2)
    draw.line([px, py-r*2, px, py+r*2], fill='yellow', width=2)

    if row['hit']:
        outcome = 'HIT'
    elif row['in_irrelevant']:
        outcome = 'DISTRACTED (irrelevant object)'
    else:
        outcome = 'BACKGROUND'
    text = f"Label:{int(row['label'])}  Pred:{int(row['pred'])}  AE:{int(row['ae'])}  -> {outcome}"
    draw.rectangle([0, 0, img.width, 28], fill='black')
    draw.text((5, 6), text, fill='white')

    img.save(os.path.join(debug_dir, row['name']))

print('Saved', len(results_df), 'debug overlays to:', debug_dir)

zip_path = OUTPUT_DIR + '/debug_overlays.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(debug_dir):
        zf.write(os.path.join(debug_dir, fname), fname)
print('Zipped to:', zip_path)


Saved 619 debug overlays to: /kaggle/working/debug_overlays
Zipped to: /kaggle/working/debug_overlays.zip
